# Semantic chunking with Jev: cut where the topic changes

Most semantic chunkers cut where the embeddings of neighbouring sentences drift apart. That measures word overlap, not whether the text moved on: a paragraph that changes vocabulary mid-argument gets split, and two sections that share vocabulary get merged.

`recipes/chunker` asks Jev, TypeSafe's decision model, a different question. Jev reads a window of numbered sentences and judges each gap: **does the next sentence start a new topic?** The answers are probabilities, so a single set of Jev requests serves every threshold, and cutting is a pure function of those probabilities plus a minimum and maximum chunk size.

This notebook runs it on QASPER, a question-answering benchmark over NLP research papers, and answers one question:

**At the same mean chunk size, do Jev's chunks retrieve the answer paragraphs better than fixed-size, recursive and embedding-similarity chunks?**

Chunk size alone moves every retrieval score, so Jev's threshold is tuned to match each baseline's mean chunk size, and each comparison is made only at that matched size.

## Which gap judge runs

- **With `OPENROUTER_API_KEY` set** (in the environment or in the repo's `.env`), Jev (`typesafe/jev-1.13`) judges every gap through OpenRouter's System One endpoint. Each request is one window of 40 numbered sentences, and the window advances 20 sentences at a time. Each gap is judged in the window where it sits nearest the centre, so Jev always sees context on both sides of it. A pilot on WANDS search results showed Jev answers per-item questions over a numbered list reliably: each item kept its answer when the list was reversed (correlation 0.98), and planted duplicates were caught at 0.95 to 0.99. A request here is about 2,900 tokens, roughly $0.00015, and the benchmark is 3,669 requests, about $0.53. The notebook judges the first 10 papers first, extrapolates the cost, and stops if the estimate exceeds `JEVQU_MAX_USD` (default $1). Answers are cached in `.jev_cache/`, so reruns cost nothing.
- **Without a key**, a **stand-in** answers the same questions: P(new topic) = 1 − cosine similarity of the two sentences' bge-small embeddings. **The stand-in is not Jev.** In that mode the results measure the pipeline and an embedding-dissimilarity chunker, and say nothing about Jev.

In [1]:
import os, sys, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
warnings.filterwarnings("ignore", message="Cannot enable progress bars")

import numpy as np
import tiktoken
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "jevqu").is_dir())
sys.path.insert(0, str(ROOT))
for line in (ROOT / ".env").read_text().splitlines() if (ROOT / ".env").exists() else []:
    key, _, value = line.partition("=")
    if key.strip() and not key.lstrip().startswith("#"):
        os.environ.setdefault(key.strip(), value.strip().strip("'\""))

from jevqu import JEV_MODEL
from jevqu.jev import Jev
from jevqu.metrics import paired_bootstrap
from recipes.chunker import qasper
from recipes.chunker.chunker import MAX_SENTENCE_CHARS, cut, gap_probs, sentences, windows

N_PAPERS = int(os.environ.get("QASPER_PAPERS", "416"))  # the whole test split; a smaller number samples it (seed 0)
WINDOW = 40        # sentences per Jev request; the window advances WINDOW // 2 sentences at a time
RETRIEVE = 5       # chunks retrieved per question
MAX_TOKENS = 512   # bge-small embeds at most 512 tokens, so Jev's chunks are capped there
WORKERS = 8
MAX_USD = float(os.environ.get("JEVQU_MAX_USD", "1"))
USE_JEV = bool(os.environ.get("OPENROUTER_API_KEY"))
JUDGE = f"Jev ({JEV_MODEL})" if USE_JEV else "stand-in (embedding dissimilarity, NOT Jev)"
print("gap judge:", JUDGE)

gap judge: Jev (typesafe/jev-1.13)


## 1. The benchmark

[QASPER](https://huggingface.co/datasets/allenai/qasper) (Dasigi et al., NAACL 2021, CC BY 4.0) pairs NLP research papers with questions about them. Researchers who had read only a paper's title and abstract wrote the questions; other researchers read the full paper, answered them, and marked the paragraphs that support each answer. Those evidence paragraphs are the references here.

The notebook uses the whole test split: 416 papers, of which 407 have at least one usable question. (Set `QASPER_PAPERS` to a smaller number to sample it, seed 0.) Each paper is one document: title, abstract, then section names and paragraphs, separated by blank lines. Evidence is pooled over annotators, and questions with no text evidence (unanswerable, or answered only by a table or figure) are dropped.

A chunker is scored by embedding its chunks locally with `BAAI/bge-small-en-v1.5` (FastEmbed), storing them in Qdrant, and retrieving 5 chunks per question from that question's paper (a payload filter on the paper id), as when asking questions about one document. Overlap is counted in characters:

- **recall**: share of the evidence text that appears in the retrieved chunks;
- **precision**: share of the retrieved text that belongs to the evidence;
- **IoU**: overlap divided by the union of retrieved and evidence text. It rewards chunks that hold the answer and little else.

In [2]:
papers, questions = qasper.load("test", N_PAPERS, seed=0)
enc = tiktoken.get_encoding("cl100k_base")
def ntok(s):
    return len(enc.encode(s, disallowed_special=()))

ptok = [ntok(t) for t in papers.values()]
etok = [sum(ntok(papers[q["paper"]][a:b]) for a, b in q["refs"]) for q in questions]
display(Markdown("| papers | questions | tokens | tokens per paper (median) | evidence paragraphs per question (mean) | evidence tokens per question (median) |\n"
                 "|---|---|---|---|---|---|\n"
                 f"| {len(papers)} | {len(questions)} | {sum(ptok):,} | {np.median(ptok):,.0f} "
                 f"| {np.mean([len(q['refs']) for q in questions]):.1f} | {np.median(etok):.0f} |"))
print("example:", questions[0]["question"])

| papers | questions | tokens | tokens per paper (median) | evidence paragraphs per question (mean) | evidence tokens per question (median) |
|---|---|---|---|---|---|
| 407 | 1297 | 1,981,968 | 4,575 | 2.5 | 220 |

example: how did they measure grammatical correctness?


## 2. Sentences and gap probabilities

Sentences are split with a regex on terminal punctuation and newlines, so every chunk is an exact slice of the paper. Each gap gets one probability, P(the next sentence starts a new topic).

In [3]:
from fastembed import TextEmbedding

embedder = TextEmbedding("BAAI/bge-small-en-v1.5")
memo = {}
def embed(texts):  # memoized: questions and repeated chunks are embedded once
    todo = [t for t in dict.fromkeys(texts) if t not in memo]
    memo.update(zip(todo, embedder.embed(todo, batch_size=64)))
    return [memo[t] for t in texts]

docs = {}
for pid, text in papers.items():
    spans = sentences(text)
    texts = [text[a:b] for a, b in spans]
    docs[pid] = {"spans": spans, "texts": texts, "sizes": [ntok(t) for t in texts]}
n_req = sum(len(windows(len(d["texts"]), WINDOW)) for d in docs.values())
print(f"{sum(len(d['texts']) for d in docs.values()):,} sentences, {n_req} windows of {WINDOW}")

class StandIn:
    # Offline stand-in with Jev's ask() contract. NOT Jev: P(new topic) = 1 - cosine of the two sentences' embeddings.
    model = "stand-in"
    def ask(self, state, questions):
        t = {s["sentence"]: s["text"] for s in state["sentences"]}
        out = {}
        for q in questions:
            k = int(q.split("_")[1])
            a, b = (v / np.linalg.norm(v) for v in embed([t[k], t[k + 1]]))
            out[q] = {"type": "noul", "noul": float(1 - a @ b)}
        return out

t0 = time.time()
judge = Jev(cache_dir=ROOT / ".jev_cache") if USE_JEV else StandIn()
probs = {}
if USE_JEV:  # price the first papers and stop before the full run if it would cost too much
    for pid in list(docs)[:10]:
        probs[pid] = gap_probs(docs[pid]["texts"], judge, WINDOW, WORKERS)
    # ponytail: estimate from uncached requests only; a pilot already in the cache estimates $0
    est = float(np.mean(judge.usage)) * n_req if judge.usage else 0.0
    print(f"pilot on {len(probs)} papers: ${sum(judge.usage):.4f} for {len(judge.usage)} requests; about ${est:.2f} for all {n_req}")
    assert est <= MAX_USD, f"estimate is above JEVQU_MAX_USD=${MAX_USD:.2f}"
for pid, d in docs.items():
    if pid not in probs:
        probs[pid] = gap_probs(d["texts"], judge, WINDOW, WORKERS)
print(f"judged {sum(map(len, probs.values())):,} gaps in {time.time() - t0:.0f}s with {JUDGE}")
if USE_JEV:
    print(f"spent ${sum(judge.usage):.4f} on {len(judge.usage)} uncached Jev requests")

77,661 sentences, 3669 windows of 40
pilot on 10 papers: $0.0000 for 0 requests; about $0.00 for all 3669


judged 77,254 gaps in 1s with Jev (typesafe/jev-1.13)
spent $0.0000 on 0 uncached Jev requests


### What the judge sees

A stretch of one paper around its most confident gap after the abstract. Each line is a sentence, preceded by P(this sentence starts a new topic).

In [4]:
pid = next(iter(docs))
p, texts = probs[pid], docs[pid]["texts"]
g = int(np.argmax(p[10:])) + 10
for i in range(max(1, g - 8), min(len(texts), g + 9)):
    mark = "<<" if i == g + 1 else "  "
    print(f"{p[i - 1]:.2f} {mark} {texts[i][:100]}")

0.11    They did not only analyze the speeches individually but also investigated agreements and disagreemen
0.05    That is, they identified references in the speech segments, determined the targets of those referenc
0.13    However, we focus only on the individual speech segments and disregard references.
0.11    For our work we have removed single-sentence speeches, HTML-tags and corrected punctuation marks.
0.07    In order to enable simple sentence splitting we replaced all sentence delimiters by a stop-token.
0.07    Furthermore, we inserted special tokens which indicate the start and the end of a speech.
0.16    Then we divided all the speeches into the four classes given by the combination of possible politica
0.10    Table TABREF1 shows the four speech classes and table TABREF2 gives a quantitative overview of the c
0.05    It can be seen that the classes RY and DN contain the majority of the speeches.
0.97 << Language Model
0.13    We use a simple statistical language model b

## 3. Baselines

All from [chonkie](https://github.com/chonkie-inc/chonkie), sizes in cl100k tokens, no overlap:

- **fixed 200**: `TokenChunker`, 200 tokens;
- **recursive**: `RecursiveChunker`, up to 200 and 400 tokens, splitting on paragraphs, then sentences, pauses and words;
- **chonkie semantic**: `SemanticChunker` with its default embedding model (`minishlab/potion-base-32M`) at similarity threshold 0.5, chunks capped at 512 tokens.

There is no fixed-400 baseline: under the 512-token cap, Jev's chunks cannot be brought to a 400-token mean (see the end of the notebook).

In [5]:
from chonkie import RecursiveChunker, SemanticChunker, TokenChunker

def spans_of(chunker):  # chonkie chunks carry their exact character offsets in the paper
    return {pid: [(c.start_index, c.end_index) for c in chunker.chunk(text)] for pid, text in papers.items()}

base_spans = {"fixed 200": spans_of(TokenChunker(tokenizer="cl100k_base", chunk_size=200, chunk_overlap=0)),
              "recursive 200": spans_of(RecursiveChunker(tokenizer="cl100k_base", chunk_size=200)),
              "recursive 400": spans_of(RecursiveChunker(tokenizer="cl100k_base", chunk_size=400)),
              "chonkie semantic, threshold 0.5": spans_of(SemanticChunker(threshold=0.5, chunk_size=MAX_TOKENS))}
base_sizes = {name: [ntok(papers[pid][a:b]) for pid, ss in s.items() for a, b in ss] for name, s in base_spans.items()}
for name, s in base_sizes.items():
    print(f"{name:32} {len(s):5} chunks, mean {np.mean(s):5.0f} tokens, max {max(s)}")

fixed 200                        10107 chunks, mean   196 tokens, max 300
recursive 200                    14093 chunks, mean   141 tokens, max 200
recursive 400                     6188 chunks, mean   320 tokens, max 400
chonkie semantic, threshold 0.5  15741 chunks, mean   127 tokens, max 691


## 4. Jev chunks at matched sizes

For each baseline, the threshold is bisected until the mean size of Jev's chunks matches the baseline's mean. The minimum chunk size is a quarter of the target and the maximum is 512 tokens. Cuts forced by the 512-token cap are counted separately: when most cuts are forced, the judge is barely deciding anything.

Each chunker's chunks go into a fresh in-memory Qdrant collection with the paper id as payload; each question retrieves its top 5 with a filter on its paper.

In [6]:
from qdrant_client import QdrantClient, models

class JevChunks:
    # Cuts the precomputed gap probabilities, no new judge calls.
    def __init__(self, threshold, min_size):
        self.threshold, self.min_size = threshold, min_size
    def starts(self, pid):
        return cut(probs[pid], docs[pid]["sizes"], self.threshold, self.min_size, MAX_TOKENS)
    def spans(self):  # {paper id: [(start, end) character spans]}
        out = {}
        for pid, d in docs.items():
            s = self.starts(pid)
            out[pid] = [(d["spans"][a][0], d["spans"][b - 1][1]) for a, b in zip(s, s[1:] + [len(d["spans"])])]
        return out
    def mean_size(self):  # sum of sentence tokens per chunk: fast, within a few tokens of ntok(chunk)
        sizes = []
        for pid, d in docs.items():
            s = self.starts(pid)
            sizes += [sum(d["sizes"][a:b]) for a, b in zip(s, s[1:] + [len(d["sizes"])])]
        return float(np.mean(sizes))
    def judged_share(self):  # share of cuts the judge made, as opposed to cuts forced by MAX_TOKENS
        cuts = [probs[pid][i - 1] >= self.threshold for pid in docs for i in self.starts(pid)[1:]]
        return float(np.mean(cuts)) if cuts else 0.0

def matched(target):
    lo, hi, min_size = 0.0, 1.0 + 1e-9, int(target // 4)
    for _ in range(40):  # mean size grows with the threshold
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if JevChunks(mid, min_size).mean_size() < target else (lo, mid)
    return JevChunks(hi, min_size)

client = QdrantClient(":memory:")
qvecs = embed([q["question"] for q in questions])

def evaluate(spans):
    # Index one chunker's chunks, retrieve RETRIEVE per question from its own paper, score against the evidence.
    if client.collection_exists("chunks"):
        client.delete_collection("chunks")
    client.create_collection("chunks", vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE))
    pts = [(pid, a, b) for pid, ss in spans.items() for a, b in ss]
    vecs = embed([papers[pid][a:b] for pid, a, b in pts])
    client.upload_points("chunks", [models.PointStruct(id=i, vector=v.tolist(), payload={"paper": pid, "span": [a, b]})
                                    for i, ((pid, a, b), v) in enumerate(zip(pts, vecs))])
    out = {"recall": [], "precision": [], "iou": [], "retrieved": []}
    for q, v in zip(questions, qvecs):
        paper = models.Filter(must=[models.FieldCondition(key="paper", match=models.MatchValue(value=q["paper"]))])
        hits = client.query_points("chunks", query=v.tolist(), query_filter=paper, limit=RETRIEVE).points
        got = [tuple(h.payload["span"]) for h in hits]
        for k, x in qasper.scores(got, q["refs"]).items():
            out[k].append(x)
        out["retrieved"].append(sum(b - a for a, b in got))
    return out

t0 = time.time()
results = []
for name, spans in base_spans.items():
    jc = matched(np.mean(base_sizes[name]))
    jspans = jc.spans()
    jsizes = [ntok(papers[pid][a:b]) for pid, ss in jspans.items() for a, b in ss]
    results.append({"name": name, "base": evaluate(spans), "jev": evaluate(jspans), "jc": jc,
                    "base_size": np.mean(base_sizes[name]), "jev_size": np.mean(jsizes), "jev_chunks": len(jsizes)})
print(f"{2 * len(results)} benchmark runs in {time.time() - t0:.0f}s")

8 benchmark runs in 2419s


## 5. Results

Each row compares one baseline with Jev's chunks at the same mean size. Changes are Jev minus the baseline, with a 95% paired-bootstrap interval over the questions.

In [7]:
label = "Jev" if USE_JEV else "stand-in"
rows = [f"| baseline | mean tokens (baseline / {label}) | recall | IoU | {label} − baseline, recall (95% CI) | {label} − baseline, IoU (95% CI) | threshold | judged cuts |",
        "|---|---|---|---|---|---|---|---|"]
for r in results:
    b, j = r["base"], r["jev"]
    dr = paired_bootstrap(b["recall"], j["recall"])
    di = paired_bootstrap(b["iou"], j["iou"])
    r["d_recall"], r["d_iou"] = dr, di
    rows.append(f"| {r['name']} | {r['base_size']:.0f} / {r['jev_size']:.0f} "
                f"| {np.mean(b['recall']):.3f} → {np.mean(j['recall']):.3f} | {np.mean(b['iou']):.4f} → {np.mean(j['iou']):.4f} "
                f"| {dr[0]:+.3f} ({dr[1]:+.3f} to {dr[2]:+.3f}) | {di[0]:+.4f} ({di[1]:+.4f} to {di[2]:+.4f}) "
                f"| {r['jc'].threshold:.3f} | {r['jc'].judged_share():.0%} |")
display(Markdown("\n".join(rows)))

| baseline | mean tokens (baseline / Jev) | recall | IoU | Jev − baseline, recall (95% CI) | Jev − baseline, IoU (95% CI) | threshold | judged cuts |
|---|---|---|---|---|---|---|---|
| fixed 200 | 196 / 199 | 0.542 → 0.599 | 0.1152 → 0.1329 | +0.058 (+0.037 to +0.077) | +0.0177 (+0.0128 to +0.0223) | 0.290 | 92% |
| recursive 200 | 141 / 142 | 0.505 → 0.532 | 0.1395 → 0.1507 | +0.027 (+0.008 to +0.047) | +0.0111 (+0.0059 to +0.0166) | 0.170 | 98% |
| recursive 400 | 320 / 323 | 0.675 → 0.739 | 0.0993 → 0.1100 | +0.064 (+0.044 to +0.086) | +0.0107 (+0.0074 to +0.0146) | 0.950 | 5% |
| chonkie semantic, threshold 0.5 | 127 / 133 | 0.465 → 0.517 | 0.1289 → 0.1520 | +0.052 (+0.032 to +0.074) | +0.0231 (+0.0169 to +0.0300) | 0.160 | 98% |

### Is the recall gain just longer retrieved chunks?

Sizes are matched on the mean, but Jev's chunks vary in length and retrieval may favour the long ones. If it did, the retrieved text would grow and precision (the share of retrieved text that belongs to the evidence) would fall. The last column compares the retrieved characters per question directly.

In [8]:
rows = [f"| baseline | precision (baseline → {label}) | {label} − baseline, precision (95% CI) | retrieved characters, {label} ÷ baseline (median per question) |",
        "|---|---|---|---|"]
for r in results:
    b, j = r["base"], r["jev"]
    dp = paired_bootstrap(b["precision"], j["precision"])
    r["d_precision"] = dp
    ratio = [rj / rb for rb, rj in zip(b["retrieved"], j["retrieved"])]
    rows.append(f"| {r['name']} | {np.mean(b['precision']):.4f} → {np.mean(j['precision']):.4f} "
                f"| {dp[0]:+.4f} ({dp[1]:+.4f} to {dp[2]:+.4f}) | {np.median(ratio):.2f} |")
display(Markdown("\n".join(rows)))

| baseline | precision (baseline → Jev) | Jev − baseline, precision (95% CI) | retrieved characters, Jev ÷ baseline (median per question) |
|---|---|---|---|
| fixed 200 | 0.1293 → 0.1458 | +0.0165 (+0.0113 to +0.0214) | 0.99 |
| recursive 200 | 0.1614 → 0.1738 | +0.0124 (+0.0064 to +0.0182) | 0.95 |
| recursive 400 | 0.1045 → 0.1148 | +0.0103 (+0.0069 to +0.0141) | 1.00 |
| chonkie semantic, threshold 0.5 | 0.1553 → 0.1778 | +0.0225 (+0.0153 to +0.0299) | 0.93 |

### Reading the results

In [9]:
lines = []
if not USE_JEV:
    lines.append("**These numbers come from the stand-in, not Jev.** They show that the pipeline runs end to end and how a chunker "
                 "cutting on adjacent-sentence embedding dissimilarity compares. Set `OPENROUTER_API_KEY` to measure Jev.")
for m in ("recall", "iou"):
    better = [r["name"] for r in results if r[f"d_{m}"][1] > 0]
    worse = [r["name"] for r in results if r[f"d_{m}"][2] < 0]
    lines.append(f"**{m}**: at matched mean size, the {label} chunks are higher than {len(better)} of {len(results)} baselines "
                 f"({', '.join(better) or 'none'}) and lower than {len(worse)} ({', '.join(worse) or 'none'}); "
                 f"the remaining intervals include zero.")
display(Markdown("\n\n".join(lines)))

**recall**: at matched mean size, the Jev chunks are higher than 4 of 4 baselines (fixed 200, recursive 200, recursive 400, chonkie semantic, threshold 0.5) and lower than 0 (none); the remaining intervals include zero.

**iou**: at matched mean size, the Jev chunks are higher than 4 of 4 baselines (fixed 200, recursive 200, recursive 400, chonkie semantic, threshold 0.5) and lower than 0 (none); the remaining intervals include zero.

## What the Jev run shows

Jev's answers were collected on 2 and 3 October 2026 with `typesafe/jev-1.13`: 3,669 requests, $0.53 in total, about 8 minutes of requests with 8 workers. Reruns read the cache and cost nothing. The 8 benchmark runs take about 40 minutes on a laptop CPU, almost all of it local embedding.

- **Jev's chunks beat every baseline on every metric.** At matched mean size, recall, precision and IoU are higher against fixed 200, recursive 200, recursive 400 and chonkie semantic, and every 95% interval is above zero. Recall gains +0.027 to +0.064; IoU +0.011 to +0.023 (8% to 18% relative); precision +0.010 to +0.023 (8% to 14%).
- **The gain is not longer retrieved text.** The retrieved text per question stays the same length (median 0.93 to 1.00 times the baseline's), so the retrieved chunks hold more of the answer, not more text.
- **Precision rises by a larger percentage than recall mostly because it starts lower** (0.10 to 0.16, against 0.47 to 0.68 for recall); in absolute terms recall gains more. The two share a numerator, the answer text retrieved, and the retrieved length barely changes, so one moves with the other.
- **Recursive 200 is the hardest baseline.** It splits on blank lines first, and QASPER's references are whole paragraphs, so its chunks often line up with the references for free; Jev sees sentences, not blank lines. Jev still wins, by the smallest margin: recall +0.027, IoU +0.011.
- **At small sizes the cutoff is low.** Matching a 130 to 200-token mean needs cutoffs of 0.16 to 0.29, so many cuts land on gaps Jev does not rate as topic changes, and the request picks the least-bad place to cut. Against recursive 400, only 5% of cuts pass the cutoff (0.95); the rest are forced by the 512-token cap, and each forced cut moves back to the likeliest gap inside the chunk, so Jev still places it.

### What this does not show

- **One domain.** English NLP research papers, with section headings and LaTeX; other kinds of text can rank the chunkers differently.
- **Answers inside one document.** Retrieval is filtered to the question's paper, so this measures finding the answer within a document, not finding the document.
- **Matched mean, not matched distribution.** Jev's chunks vary in size and fixed ones do not, so with 5 chunks retrieved per question the two can retrieve different amounts of text (here the medians are within 7%).
- **One embedding model.** bge-small truncates at 512 tokens, so every chunker is capped near that, and a stronger embedding model can change the ranking.
- **No overlap.** Every chunker cuts without overlap; overlapping windows help fixed-size chunkers most.